In [4]:
"""
generate_data.py
-----------------
Synthesizes a realistic 2-echelon supply chain dataset since no real
company data is available for this project.

NETWORK STRUCTURE (2-echelon = 2 layers of nodes):
    3 Suppliers  -->  1 Central Warehouse  -->  5 Retail Stores

We generate:
  1. daily_demand.csv     -> demand at each of the 5 stores, 2 years of history
  2. supplier_capacity.csv -> daily supply capacity from each of the 3 suppliers
  3. network_config.csv   -> lead times, costs, safety stock params per node

Demand = trend + weekly seasonality + yearly seasonality + random noise
This mimics real retail demand (e.g. weekend spikes, holiday season bump).
"""
import os
import numpy as np
import pandas as pd

np.random.seed(42)

# ---------------------------------------------------------------------------
# PATH SETUP - for Jupyter notebooks, __file__ doesn't exist, so set the
# project root directly. Change this ONE line to match where you put the
# project folder on your PC.
# ---------------------------------------------------------------------------
PROJECT_ROOT = r"C:\path\to\your\supply_chain_project"   # <-- EDIT THIS
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

START_DATE = "2023-01-01"
N_DAYS = 730  # 2 years of daily data
STORES = [f"Store_{i}" for i in range(1, 6)]
SUPPLIERS = ["Supplier_A", "Supplier_B", "Supplier_C"]

dates = pd.date_range(START_DATE, periods=N_DAYS, freq="D")

def build_demand_series(base_level, weekend_boost, yearly_amp, noise_sd, growth_per_day):
    t = np.arange(N_DAYS)
    trend = base_level + growth_per_day * t
    weekly = weekend_boost * (pd.Series(dates.dayofweek).isin([5, 6]).astype(int).to_numpy())
    yearly = yearly_amp * np.sin(2 * np.pi * t / 365.25 + np.pi / 2)  # peak mid-year
    noise = np.random.normal(0, noise_sd, N_DAYS)
    series = trend + weekly + yearly + noise
    return np.clip(series, a_min=0, a_max=None).round(0)

# Each store gets its own demand profile (different size/volatility)
store_params = {
    "Store_1": dict(base_level=120, weekend_boost=25, yearly_amp=20, noise_sd=8, growth_per_day=0.02),
    "Store_2": dict(base_level=90,  weekend_boost=15, yearly_amp=15, noise_sd=6, growth_per_day=0.01),
    "Store_3": dict(base_level=150, weekend_boost=35, yearly_amp=30, noise_sd=10, growth_per_day=0.03),
    "Store_4": dict(base_level=70,  weekend_boost=10, yearly_amp=10, noise_sd=5, growth_per_day=0.00),
    "Store_5": dict(base_level=110, weekend_boost=20, yearly_amp=18, noise_sd=7, growth_per_day=0.015),
}

demand_rows = []
for store, params in store_params.items():
    series = build_demand_series(**params)
    for d, v in zip(dates, series):
        demand_rows.append({"date": d, "store": store, "demand_units": int(v)})

demand_df = pd.DataFrame(demand_rows)
demand_df.to_csv(os.path.join(OUTPUTS_DIR, "daily_demand.csv"), index=False)

# Supplier daily capacity (units they can ship to the central warehouse per day)
supplier_base_capacity = {"Supplier_A": 250, "Supplier_B": 200, "Supplier_C": 180}
capacity_rows = []
for supplier, base_cap in supplier_base_capacity.items():
    noise = np.random.normal(0, base_cap * 0.05, N_DAYS)
    cap = np.clip(base_cap + noise, a_min=0, a_max=None).round(0)
    for d, v in zip(dates, cap):
        capacity_rows.append({"date": d, "supplier": supplier, "capacity_units": int(v)})

capacity_df = pd.DataFrame(capacity_rows)
capacity_df.to_csv(os.path.join(OUTPUTS_DIR, "supplier_capacity.csv"), index=False)

# Network configuration: lead times & costs (assumed, documented as assumptions)
network_config = pd.DataFrame([
    {"node": "Supplier_A", "type": "supplier", "normal_lead_time_days": 4, "unit_cost": 8.5, "share_of_supply": 0.38},
    {"node": "Supplier_B", "type": "supplier", "normal_lead_time_days": 6, "unit_cost": 7.9, "share_of_supply": 0.31},
    {"node": "Supplier_C", "type": "supplier", "normal_lead_time_days": 5, "unit_cost": 8.2, "share_of_supply": 0.28},
    {"node": "Central_Warehouse", "type": "warehouse", "normal_lead_time_days": 1, "unit_cost": 0.5, "share_of_supply": np.nan},
])
network_config.to_csv(os.path.join(OUTPUTS_DIR, "network_config.csv"), index=False)

print("Generated:")
print(f" - daily_demand.csv       ({len(demand_df)} rows, {len(STORES)} stores, {N_DAYS} days)")
print(f" - supplier_capacity.csv  ({len(capacity_df)} rows, {len(SUPPLIERS)} suppliers)")
print(f" - network_config.csv     ({len(network_config)} rows)")
print("\nSample demand data:")
print(demand_df.head())

Generated:
 - daily_demand.csv       (3650 rows, 5 stores, 730 days)
 - supplier_capacity.csv  (2190 rows, 3 suppliers)
 - network_config.csv     (4 rows)

Sample demand data:
        date    store  demand_units
0 2023-01-01  Store_1           169
1 2023-01-02  Store_1           139
2 2023-01-03  Store_1           145
3 2023-01-04  Store_1           152
4 2023-01-05  Store_1           138
